# Fish detection - unattended training on Kaggle

Runs the training pipeline headless for up to ~10.5 hours, surviving crashes and
resuming from the previous session's checkpoints.

## Before you run this

1. **Settings -> Accelerator -> GPU T4 x2** (or P100).
2. **Settings -> Internet -> On** (needs phone verification on a new account).
3. **Add-ons -> Secrets -> Add secret** named `GIT_TOKEN`, holding a GitHub
   personal access token with `repo` scope. Required because the repo is private.
   Leave it unset if you make the repo public.
4. To *continue* a previous session: **Data -> Add Input -> Your Work -> Notebook
   Output**, and pick this notebook's previous version. Checkpoints are restored
   automatically and training resumes instead of restarting.

## To run it

**Save Version -> Save & Run All (Commit)**. Close the tab; it runs on Kaggle's
machines. Do *not* leave it as an interactive session, which stops when idle.

Results land in `/kaggle/working/kaggle_report.json`.

## 1. Configure

The only cell worth editing.

In [ ]:
CONFIG     = "yolov8n_960"
SEEDS      = [2]
MAX_HOURS  = 10.5

REPO_OWNER = "TDuong04"
REPO_NAME  = "image-fish-AI"
BRANCH     = "main"

## 2. Fetch the project

The token is read from Kaggle Secrets and used only to build the clone URL, so it
never appears in the notebook source or the committed output log.

In [ ]:
import subprocess, sys, os, pathlib

token = None
try:
    from kaggle_secrets import UserSecretsClient
    token = UserSecretsClient().get_secret("GIT_TOKEN")
    print("GIT_TOKEN found")
except Exception as exc:
    print(f"No GIT_TOKEN secret ({type(exc).__name__}). Assuming a public repo.")

host = "github.com"
repo_url = (f"https://{token}@{host}/{REPO_OWNER}/{REPO_NAME}.git" if token
            else f"https://{host}/{REPO_OWNER}/{REPO_NAME}.git")

target = pathlib.Path("/kaggle/working") / REPO_NAME
if target.exists():
    print(f"{target} already present")
else:
    r = subprocess.run(["git", "clone", "--depth", "1", "-b", BRANCH, repo_url, str(target)],
                       capture_output=True, text=True)
    # Never echo the URL on failure -- it carries the token.
    print("clone ok" if r.returncode == 0 else
          f"CLONE FAILED (exit {r.returncode}). Check GIT_TOKEN scope and repo name.")
    if r.returncode != 0:
        raise SystemExit("clone failed")

print(sorted(p.name for p in target.iterdir())[:12])

## 3. Train

Everything below is driven by `kaggle/kaggle_train.py` in the repo: dependency
install, dataset download and preparation, checkpoint restore, supervised
training with crash recovery, and the final report.

Output is long. The summary at the end is the part to read.

In [ ]:
cmd = [sys.executable, str(target / "kaggle" / "kaggle_train.py"),
       "--config", CONFIG,
       "--seeds", *[str(s) for s in SEEDS],
       "--max-hours", str(MAX_HOURS),
       "--skip-clone"]

proc = subprocess.Popen(cmd, cwd=str(target), stdout=subprocess.PIPE,
                        stderr=subprocess.STDOUT, text=True, bufsize=1)
for line in proc.stdout:
    print(line, end="")
proc.wait()
print(f"\nexit code: {proc.returncode}")

## 4. Results

`kaggle_report.json` is the one file to read. If `outcome` is `budget_reached`,
attach this version's output to the next run and it continues from here.

In [ ]:
import json, shutil, pathlib

report_path = pathlib.Path("/kaggle/working/kaggle_report.json")
if report_path.exists():
    report = json.loads(report_path.read_text())
    print(f"outcome : {report.get('outcome')}")
    print(f"gpu     : {report.get('gpu')}")
    print(f"restored: {report.get('restored_runs')} previous run(s)")
    for name, m in (report.get("metrics") or {}).items():
        if "map50" in m:
            print(f"  {name}: mAP50={m['map50']:.4f}  mAP50-95={m['map50_95']:.4f}")
    for f in report.get("failures") or []:
        print(f"  FAILURE {f['failure_kind']} (retryable={f['retryable']}) "
              f"on seed {f['seed']} attempt {f['attempt']}")
else:
    print("No report written -- the training cell did not reach the end.")

# Keep checkpoints and run provenance in the output; drop the dataset copy,
# which is 1.2 GB of re-downloadable data and would crowd the 20 GB limit.
data_dir = pathlib.Path("/kaggle/working") / REPO_NAME / "data"
if data_dir.exists():
    shutil.rmtree(data_dir, ignore_errors=True)
    print("removed data/ from the output")

total = sum(f.stat().st_size for f in pathlib.Path("/kaggle/working").rglob("*") if f.is_file())
print(f"output size: {total / 1e9:.2f} GB (limit 20 GB)")